# U09_C2 — Secondary Structure Prediction and Scoring
**Course:** Concepts of Bioinformatics (CBBI-501) · **Unit 9:** Predictive Methods using Protein Sequence
**Type:** Course notebook · **Time:** 40 min · **Difficulty:** ★★☆
**Goes with:** Lectures L25, Practical S11 (GOR IV & PSIPRED)
**Before this:** U09_C1

## 1. What you will learn
- **Explain** 3-state secondary structure prediction ($H = \text{alpha-helix}, E = \text{beta-strand}, C = \text{coil}$) [LO6].
- **Compare** statistical propensity methods (Chou–Fasman, GOR IV) with modern machine-learning models (PSIPRED) [LO6].
- **Calculate** the standard $Q_3$ accuracy score against experimentally determined crystallographic structures (DSSP) [LO6].
- **Construct** a $3 \times 3$ confusion matrix to measure per-state prediction sensitivity and specificity [LO6].


## 2. Biological question
Before AlphaFold, determining protein structure required predicting local secondary structure from sequence alone. How accurate are algorithms like GOR IV (information theory) and PSIPRED (neural networks)? By comparing predicted secondary structure strings for human *HBB* with the true crystallographic secondary structure derived from the 2.0 Å crystal structure (PDB `1A3N`), how do we quantify prediction accuracy ($Q_3$)?


In [ ]:
import sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "biopython"], check=True)

NOTEBOOK_ID = "U09_C2"
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").is_dir())
DATA_DIR = ROOT / "data"
OUT_DIR = ROOT / "outputs" / NOTEBOOK_ID
OUT_DIR.mkdir(parents=True, exist_ok=True)

import Bio
print("Python", sys.version.split()[0], "| Biopython", Bio.__version__)

## 3. Concept 1: The Three Secondary Structure States
### Theory
Although crystallography assigns secondary structure into 8 DSSP states ($H, G, I, E, B, T, S, C$), secondary structure prediction simplifies this into a **3-state classification**:
- **`H` (Helix)**: $\alpha$-helices (DSSP $H$) and $3_{10}$-helices ($G$).
- **`E` (Strand)**: Extended $\beta$-strands ($E$) and isolated $\beta$-bridges ($B$).
- **`C` (Coil / Loop)**: Turns ($T$), bends ($S$), and unstructured coils ($C$).

Hemoglobin beta is an **all-$\alpha$** protein consisting of 8 classic helices (A through H) connected by short coils.


## 4. Concept 2: Inspecting GOR IV, PSIPRED, and Real DSSP Strings
### Theory
- **GOR IV**: Uses statistical information theory based on amino acid propensities in a 17-residue local window (accuracy $\approx 64\%$).
- **PSIPRED**: Uses position-specific scoring matrices (PSSMs from PSI-BLAST) fed into a two-stage feedforward neural network (accuracy $\approx 80\%$).
- **DSSP**: Definitive secondary structure assigned from atomic 3D coordinates by identifying hydrogen bonds in PDB crystal structures.


In [ ]:
# Worked Example: Loading prediction strings from data/expected/
def load_sec_str(filename):
    with open(DATA_DIR / "expected" / filename) as f:
        for line in f:
            line = line.strip()
            # If line is only letters H, E, C
            if line and set(line).issubset(set("HECT ")):
                return line.replace(" ", "")
    return ""

gor_pred = load_sec_str("gor4_hbb_secondary_structure.txt")
psi_pred = load_sec_str("psipred_hbb_secondary_structure.txt")
dssp_raw = load_sec_str("dssp_hbb_1a3n.txt")

# Standardize DSSP 8-state to 3-state (H=H, E=E, else C)
dssp_true = "".join("H" if c in "HG" else ("E" if c == "E" else "C") for c in dssp_raw)

print(f"GOR IV Length:  {len(gor_pred)} | Start: {gor_pred[:30]}")
print(f"PSIPRED Length: {len(psi_pred)} | Start: {psi_pred[:30]}")
print(f"DSSP True Length: {len(dssp_true)} | Start: {dssp_true[:30]}")


## 5. Concept 3: Computing Q3 Accuracy
### Theory
The standard evaluation metric in secondary structure prediction is the **three-state overall accuracy ($Q_3$)**:
$$Q_3 = \frac{\text{Number of correctly predicted residues}}{\text{Total number of residues}} \times 100 = \frac{N_H + N_E + N_C}{N_{\text{total}}} \times 100$$
where $N_H, N_E, N_C$ are the counts of residues where prediction matches the true DSSP state.


In [ ]:
# Worked Example: Calculating Q3 Accuracy
min_len = min(len(gor_pred), len(psi_pred), len(dssp_true))

gor_matches = sum(1 for i in range(min_len) if gor_pred[i] == dssp_true[i])
psi_matches = sum(1 for i in range(min_len) if psi_pred[i] == dssp_true[i])

q3_gor = (gor_matches / min_len) * 100
q3_psi = (psi_matches / min_len) * 100

print(f"=== Three-State Accuracy (Q3) for Human HBB ===")
print(f"Total residues evaluated: {min_len}")
print(f"GOR IV Correct:  {gor_matches:3} / {min_len} -> Q3 = {q3_gor:.2f}%")
print(f"PSIPRED Correct: {psi_matches:3} / {min_len} -> Q3 = {q3_psi:.2f}%")


In [ ]:
# Worked Example: Confusion Matrix for PSIPRED
states = ["H", "E", "C"]
confusion = {s_true: {s_pred: 0 for s_pred in states} for s_true in states}

for i in range(min_len):
    t = dssp_true[i]
    p = psi_pred[i]
    if t in confusion and p in confusion[t]:
        confusion[t][p] += 1

print("=== Confusion Matrix (Rows = Actual DSSP, Columns = PSIPRED Predicted) ===")
print(f"{'Actual':8} | {'Pred H':8} | {'Pred E':8} | {'Pred C':8}")
print("-" * 42)
for s_true in states:
    print(f"{s_true:8} | {confusion[s_true]['H']:8} | {confusion[s_true]['E']:8} | {confusion[s_true]['C']:8}")


## 6. Summary
- Secondary structure prediction standardizes local conformations into 3 states: Helix ($H$), Sheet ($E$), Coil ($C$).
- Experimental validation compares predictions with crystallographic DSSP assignments.
- $Q_3$ accuracy measures overall percentage concordance.
- Neural network methods incorporating evolutionary PSSMs (PSIPRED) outperform single-sequence propensity models (GOR).


## 7. Next Steps
- Complete protein prediction tasks in `U09_P_Protein_Prediction.ipynb`.
- Self-test secondary structure concepts in `U09_Q_Predictive_Protein.ipynb`.
